In [ ]:
!pip install datasets

In [ ]:
from google.colab import files
import json
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSeq2SeqLM

# Upload train/test JSON files generated by split_train_test.py
print("Please upload your TRAIN JSON file:")
uploaded_train = files.upload()
train_file = next(iter(uploaded_train.keys()))

print("Please upload your TEST JSON file:")
uploaded_test = files.upload()
test_file = next(iter(uploaded_test.keys()))

# Load only the samples array from each file (files contain metadata + samples)
with open(train_file, "r", encoding="utf-8") as f:
    train_meta = json.load(f)
with open(test_file, "r", encoding="utf-8") as f:
    test_meta = json.load(f)

train_df = pd.DataFrame(train_meta["samples"])
test_df = pd.DataFrame(test_meta["samples"])

# Convert to HuggingFace Datasets
dataset_train = Dataset.from_pandas(train_df)
dataset_test = Dataset.from_pandas(test_df)

# Split test into validation/test halves if desired
split = dataset_test.train_test_split(test_size=0.5, seed=42)
dataset_validation = split["train"]
dataset_test = split["test"]

dataset_dict = {
    "train": dataset_train,
    "validation": dataset_validation,
    "test": dataset_test
}

# Tokenization
tokenizer = AutoTokenizer.from_pretrained("t5-small")
label_pad_token_id = tokenizer.pad_token_id
max_length = 128  # adjust upward if needed

def preprocess_function(batch):
    inputs = tokenizer(
        batch["input"],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["output"],
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(token if token != label_pad_token_id else -100) for token in seq]
        for seq in labels_ids
    ]
    inputs["labels"] = labels_ids
    return inputs

dataset_train = dataset_dict["train"].map(preprocess_function, batched=True)
dataset_validation = dataset_dict["validation"].map(preprocess_function, batched=True)
dataset_test = dataset_dict["test"].map(preprocess_function, batched=True)

# TrainingArguments
training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=50,
    report_to=[]
)
# Model and trainer
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_validation,
    tokenizer=tokenizer
    # processing_class=tokenizer  # optional to silence the future warning
)

trainer.train()

Please upload your TRAIN JSON file:


Saving dataset_1_operation_train.json to dataset_1_operation_train.json
Please upload your TEST JSON file:


Saving dataset_1_operation_test.json to dataset_1_operation_test.json


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipython-input-740085233.py:82: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,1.376000
1000,0.656300
1500,0.510900
2000,0.414900
2500,0.370600
3000,0.306400
3500,0.295700
4000,0.267700
4500,0.237600
5000,0.230400


TrainOutput(global_step=33900, training_loss=0.1580272137802259, metrics={'train_runtime': 2086.2052, 'train_samples_per_second': 129.853, 'train_steps_per_second': 16.25, 'total_flos': 9166023504691200.0, 'train_loss': 0.1580272137802259, 'epoch': 50.0})

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ce20310c16ed5eea54113a6b2c5bba7d848924520e5aa365bb21fd1c3e9f20fc
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import os
import torch
from evaluate import load
import editdistance
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# --- helpers ---------------------------------------------------------------

def get_model_size(path):
    total_size = 0
    for dirpath, _, filenames in os.walk(path):
        for fname in filenames:
            total_size += os.path.getsize(os.path.join(dirpath, fname))
    return total_size / (1024 * 1024)  # MB

def compute_levenshtein_similarity(predictions, references):
    sims = []
    for pred, ref in zip(predictions, references):
        pred = pred.strip()
        ref = ref.strip()
        dist = editdistance.eval(pred, ref)
        max_len = max(len(pred), len(ref))
        sims.append(1.0 - dist / max_len if max_len > 0 else 1.0)
    return sum(sims) / len(sims)

def compute_bleu_score(predictions, references):
    smoothing = SmoothingFunction().method1
    scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = pred.strip().split()
        ref_tokens = ref.strip().split()
        if ref_tokens:
            scores.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing))
        else:
            scores.append(0.0)
    return sum(scores) / len(scores)

def compute_rouge_scores(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    rouge1, rougeL = [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref.strip(), pred.strip())
        rouge1.append(scores["rouge1"].fmeasure)
        rougeL.append(scores["rougeL"].fmeasure)
    return {
        "rouge1": sum(rouge1) / len(rouge1),
        "rougeL": sum(rougeL) / len(rougeL),
    }

# --- prediction loop ------------------------------------------------------

predictions = []
references = []

model.eval()
for example in dataset_test:
    # Prepare inputs
    input_ids = torch.tensor(example["input_ids"]).unsqueeze(0).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=128)

    # Decode generated text
    pred_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    predictions.append(pred_text)

    # Decode reference text (drop masked pads = -100)
    label_ids = example["labels"]
    if isinstance(label_ids, torch.Tensor):
        label_ids = label_ids.tolist()
    label_ids = [tok for tok in label_ids if tok != -100]
    ref_text = tokenizer.decode(label_ids, skip_special_tokens=True).strip()
    references.append(ref_text)

# --- metrics --------------------------------------------------------------

exact_match = load("exact_match")
em_result = exact_match.compute(predictions=predictions, references=references)["exact_match"]

ls_result = compute_levenshtein_similarity(predictions, references)
bleu_result = compute_bleu_score(predictions, references)
rouge_results = compute_rouge_scores(predictions, references)

results = pd.DataFrame([{
    "Model": "t5-small-finetuned",
    "Exact Match": em_result,
    "Levenshtein Similarity": ls_result,
    "BLEU Score": bleu_result,
    "ROUGE-1": rouge_results["rouge1"],
    "ROUGE-L": rouge_results["rougeL"],
    "Model Size (MB)": get_model_size(".")
}])

print(results)

                Model  Exact Match  Levenshtein Similarity  BLEU Score  \
0  t5-small-finetuned     0.818318                0.955798     0.34855   

    ROUGE-1   ROUGE-L  Model Size (MB)  
0  0.949929  0.948168     50829.998594  


In [ ]:
import os
import re

def get_last_checkpoint(output_dir):
    checkpoints = []
    for name in os.listdir(output_dir):
        if name.startswith("checkpoint-"):
            # extract the number
            num = int(name.split("-")[-1])
            checkpoints.append((num, os.path.join(output_dir, name)))
    if not checkpoints:
        return None
    # return highest-numbered checkpoint
    return max(checkpoints, key=lambda x: x[0])[1]

last_ckpt = get_last_checkpoint("output")
print("Last checkpoint:", last_ckpt)


Last checkpoint: output/checkpoint-33900


In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the source and destination paths
source_checkpoint_dir = last_ckpt
destination_drive_path = '/content/drive/MyDrive/t5_small_fp32_finetuned(50)(above80%)'

# Create the destination directory if it doesn't exist
os.makedirs(destination_drive_path, exist_ok=True)

# Copy the checkpoint directory to Google Drive
try:
    shutil.copytree(source_checkpoint_dir, destination_drive_path, dirs_exist_ok=True)
    print(f"Successfully copied checkpoint from '{source_checkpoint_dir}' to '{destination_drive_path}'")
except Exception as e:
    print(f"Error copying checkpoint: {e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully copied checkpoint from 'output/checkpoint-33900' to '/content/drive/MyDrive/content/drive/MyDrive/t5_small_fp32_finetuned(50)(above80%)'


In [1]:
!pip install optimum[onnxruntime]

In [2]:
import os
from google.colab import drive
from optimum.exporters.onnx import main_export


# Mount Google Drive
drive.mount('/content/drive')

# Google Drive path to your fine‑tuned model folder
MODEL_PATH = "/content/drive/MyDrive/t5_small_fp32_finetuned(50)(above80%)"

# ONNX output directory (inside the same folder, like before)
ONNX_OUTPUT_DIR = os.path.join(MODEL_PATH, "onnx")

os.makedirs(ONNX_OUTPUT_DIR, exist_ok=True)

print(f"Exporting ONNX to: {ONNX_OUTPUT_DIR}")

main_export(
    model_name_or_path=MODEL_PATH,
    output=ONNX_OUTPUT_DIR,
    task="text2text-generation-with-past",
    opset=14,
    decoder=True,
    merge_decoder=True,
)

print("✓ ONNX export finished.")
print("ONNX files are now in:", ONNX_OUTPUT_DIR)

Multiple distributions found for package optimum. Picked distribution: optimum-onnx


Mounted at /content/drive
Exporting ONNX to: /content/drive/MyDrive/t5_small_fp32_finetuned(50)(above80%)/onnx


Opset 14 is lower than the recommended minimum opset (18) to export t5. The ONNX export may fail or the exported model may be suboptimal.
/usr/local/lib/python3.12/dist-packages/transformers/models/t5/modeling_t5.py:1263: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py:108: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.keys is None or self.keys.numel() == 0:
Could not find ONNX initializer for torch parameter decoder.embed_tokens.weight. decoder.embed_tok

✓ ONNX export finished.
ONNX files are now in: /content/drive/MyDrive/t5_small_fp32_finetuned(50)(above80%)/onnx
